In [1]:
%pip install pandas geopandas plotly scikit-learn numpy
import pandas as pd
import geopandas as gpd
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import numpy as np


[notice] A new release of pip available: 22.2.2 -> 25.0
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
def load_and_process_data():
    # Load with proper delimiter
    energy_df = pd.read_csv('energy-and-utilities-linc.csv', delimiter=',', encoding='utf-8')
    
    # Print data structure for debugging
    print("Columns:", energy_df.columns.tolist())
    print("\nFirst few rows:")
    print(energy_df.head())
    
    # Clean column names
    energy_df.columns = energy_df.columns.str.strip().str.replace(' ', '_')
    
    # Verify required columns exist
    required_cols = ['Area_Name', 'Year', 'Variable', 'Value']
    missing_cols = [col for col in required_cols if col not in energy_df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Load county coordinates
    counties_gdf = pd.read_csv('NCCountyCoordinates.csv')
    
    return energy_df, counties_gdf

In [3]:
# 2. Model Training
def train_prediction_model(energy_df):
    # Prepare features and target
    X = energy_df[['Year']]
    y = energy_df['Occupied Housing Units Heated by Electricity']
    
    model = RandomForestRegressor(n_estimators=100)
    model.fit(X, y)
    return model

In [4]:

# 3. Generate Future Predictions
def generate_predictions(model, counties):
    future_years = [2025, 2030]
    predictions = {}
    
    for year in future_years:
        pred = model.predict([[year]])
        predictions[year] = pred
        
    return predictions

In [5]:
# 4. Create Interactive Map
def create_choropleth_map(energy_df, counties_gdf, predictions):
    # Combine historical and predicted data
    years = [1990, 2000, 2010, 2015, 2020, 2025, 2030]
    
    # Create figure
    fig = px.choropleth(
        energy_df,
        geojson=counties_gdf,
        locations='Area Name',
        featureidkey='properties.NAME',
        color='Value',
        animation_frame='Year',
        range_color=(0, energy_df['Value'].max()),
        scope="usa",
        title='NC Energy Consumption by County (1990-2030)',
        labels={'Value': 'Energy Consumption'}
    )

    # Configure map view
    fig.update_geos(
        fitbounds="locations",
        visible=False,
        center={"lat": 35.5, "lon": -80},
        scope='usa',
    )

    # Add slider
    fig.update_layout(
        sliders=[{
            'active': 0,
            'steps': [
                {
                    'method': 'animate',
                    'label': str(year),
                    'args': [[str(year)]]
                } for year in years
            ]
        }]
    )

    return fig

In [6]:
# Main execution
def main():
    # Load and process data
    energy_df, counties_gdf = load_and_process_data()
    
    # Train model
    model = train_prediction_model(energy_df)
    
    # Generate predictions
    predictions = generate_predictions(model, counties_gdf)
    
    # Create visualization
    fig = create_choropleth_map(energy_df, counties_gdf, predictions)
    
    # Show interactive map
    fig.show()

if __name__ == "__main__":
    main()

ParserError: Error tokenizing data. C error: Expected 1 fields in line 4, saw 4
